# Week 1 · Day 1 — What Is AI? From Rules to Learning

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*

**Facilitator:** Solomon Wilson MBCS | PhD Student, Computer Science | Deputy HOD Transport Planning & Operations | HOD, IT & Audit Supervisor, SLPTA

**Mode:** Google Colab (zero-install)

**Mental model layer:** L1 — Rules vs. Learning

**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay


**Module:** 1 · **Week:** 1 · **Tier:** Intro  
**New concept:** AI is pattern recognition — machines find patterns in data to make predictions  
**Deliverable wired in:** None

## Learning objectives
By the end of today you will be able to:
- Explain, in plain language, the difference between a **rule-based system** (we write the rule) and a **learning system** (the computer finds the rule from examples).
- Give one SLPTA example each of **narrow**, **general**, and **generative** AI.
- Decide when AI should **not** be trusted to make a dispatch decision.

## Why this matters for SLPTA

This morning the **R12** service (Wilberforce → CBD) left **25 minutes late**, and the dispatch desk received dozens of complaints. Someone has to read each complaint and decide: *is this urgent (a safety issue) or routine (a timetable question)?*

Doing that by hand does not scale on a busy day. Before we ask a computer to help, we must answer a first-principles question: **can a computer decide the rule for "urgent" by itself, or must a human write that rule down?** That is the whole difference between traditional programming and AI.

## Environment setup

In [1]:
# Today's lab uses pandas (preinstalled in Colab). We also install google-genai
# so that the SAME bootstrap import line works in every notebook this course —
# even though we will not call a language model until Day 9.
!pip install -q google-genai
print("Environment ready.")

Environment ready.


In [ ]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path

# Find the folder that holds shared/slpta_bootstrap.py, wherever the repo sits.
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()          # generates course_data/ on first run (no-op after)
print("Model configured:", MODEL)
print(load_route12_context())

## Concept — first principles

**Traditional programming:** a human writes the rule, the computer follows it.
> *Input + Rule → Output*

**Machine learning (AI):** a human gives the computer many labelled **examples**, and the computer works out the rule itself.
> *Input + Output (examples) → Rule*

That flipped arrow is the entire idea. Everything else this course adds — neural networks, language models — is a more powerful way to learn the rule from examples.

**Three kinds of AI you will hear about:**

| Type | What it means | SLPTA example |
|------|---------------|---------------|
| **Narrow** | Does one specific task | Flagging trips likely to be delayed |
| **General** | Human-level across *any* task | Does not exist yet |
| **Generative** | Produces new content (text, images) | Drafting a passenger apology SMS |

*Jargon, defined once:* a **label** is the correct answer attached to an example (e.g. a complaint labelled "Urgent").

<p align="center"></p>

<!-- cell-diagram:c09 -->
<p align="center"></p>

### Check your understanding (before running)
We are about to show you 10 real-looking passenger complaints. **You** will label each as `Urgent` or `Routine` — you are acting as the rule. Then a simple keyword rule will label the same 10.

**Predict:** out of 10, how many do you think a plain keyword rule will get right? Write your guess down before running.

In [4]:
import pandas as pd

# 1. DATA INGESTION
# Load the synthetic passenger complaints dataset using the course's bootstrap loader.
# This dataset simulates feedback regarding SLPTA transport services.
complaints = load_complaints()

# 2. DETERMINISTIC SAMPLING
# Extract a fixed sample of 10 complaints for manual review/labelling exercises.
# - random_state=7: Ensures the same 'random' selection across all developer environments.
# - reset_index(drop=True): Resets the dataframe index to 0-9 for clean display and referencing.
sample = complaints.sample(10, random_state=7).reset_index(drop=True)

# 3. DISPATCHER VIEW SELECTION
# Filter the sample to show only the relevant fields a human dispatcher would evaluate:
# - ticket_id: Unique record identifier.
# - raw_text: The actual passenger message text.
# - severity: The preliminary system-assigned severity level (Low, Medium, High).
sample[["ticket_id", "raw_text", "severity"]]


,ticket_id,raw_text,severity
0,C0266,R7 never showed at the scheduled time again.,Medium
1,C0066,"Bus on R7 was 47 minutes late this morning, no...",Low
2,C0121,Is R12 running during the public holiday?,Low
3,C0133,The R7 vehicle had bald tyres and a broken door.,High
4,C0379,"Waited almost an hour for R9, completely unacc...",Low
5,C0027,Driver on R9 was speeding and overtaking dange...,High
6,C0288,R14 never showed at the scheduled time again.,Low
7,C0112,The R12 vehicle had bald tyres and a broken door.,High
8,C0281,Conductor on R3 was rude and shouting at an el...,Low
9,C0165,Why did I pay 1865 Le when the board says the ...,High


In [6]:
display(complaints)

,ticket_id,timestamp,channel,route_id,raw_text,category,severity,is_urgent,resolved
0,C0001,2025-03-14 12:08,SMS,R7,R7 never showed at the scheduled time again.,Delay,Low,False,True
1,C0002,2025-03-24 23:20,WhatsApp,R12,Rubbish all over the floor of the R12 vehicle.,Cleanliness,High,True,False
2,C0003,2025-02-09 02:35,WhatsApp,R12,The R12 vehicle had bald tyres and a broken door.,Safety,High,True,True
3,C0004,2025-02-22 07:02,SMS,R3,R3 never showed at the scheduled time again.,Delay,Medium,False,True
4,C0005,2025-02-17 13:40,SMS,R7,Operator OP-103 is collecting extra money and ...,Overcharging,Low,False,False
...,...,...,...,...,...,...,...,...,...
395,C0396,2025-02-28 18:05,SMS,R21,"I left my phone on the R21 bus this afternoon,...",Lost Item,Medium,True,True
396,C0397,2025-03-03 12:16,Call,R12,Conductor on R12 was rude and shouting at an e...,Staff Conduct,Low,False,False
397,C0398,2025-02-18 08:21,SMS,R9,Where can I get the timetable for R9?,Other,Low,False,True
398,C0399,2025-03-12 17:27,WhatsApp,R14,Operator OP-110 let too many standing passenge...,Safety,High,True,False


<!-- cell-diagram:c11 -->
<p align="center"></p>

### Check your understanding (before running)
You will label 10 complaints as **Urgent** or **Routine** before the rule runs.

**Predict:** How many of the 10 will match what the keyword rule picks for Route **R12** complaints?

In [ ]:
# TODO: replace every "___" with either "Urgent" or "Routine".
my_labels = [
    "___",
    "___",
    "___",
    "___",
    "___",
    "___",
    "___",
    "___",
    "___",
    "___",
]

# A reference labelling (severity High, or an inherently risky category, = Urgent).
gold = sample["is_urgent"].map({True: "Urgent", False: "Routine"}).tolist()

if "___" in my_labels:
    print("Fill in your 10 labels above (replace each ___), then re-run this cell.")
else:
    correct = sum(a.strip().title() == g for a, g in zip(my_labels, gold))
    print(f"You matched the reference on {correct} of {len(gold)} complaints.")

Fill in your 10 labels above (replace each ___), then re-run this cell.


In [ ]:
# TODO: replace every "___" with either "Urgent" or "Routine".
my_labels = [
    "Routine",  # 0: Scheduled time complaint
    "Routine",  # 1: Delay complaint
    "Routine",  # 2: General inquiry
    "Urgent",   # 3: Bald tyres/broken door (Safety)
    "Routine",  # 4: Wait time complaint
    "Urgent",   # 5: Speeding/dangerous (Safety)
    "Routine",  # 6: Scheduled time complaint
    "Urgent",   # 7: Bald tyres (Safety)
    "Routine",  # 8: Rude staff
    "Urgent",   # 9: Pricing/Overcharging (Integrity)
]

# A reference labelling (severity High, or an inherently risky category, = Urgent).
gold = sample["is_urgent"].map({True: "Urgent", False: "Routine"}).tolist()

if "___" in my_labels:
    print("Fill in your 10 labels above (replace each ___), then re-run this cell.")
else:
    correct = sum(a.strip().title() == g for a, g in zip(my_labels, gold))
    print(f"You matched the reference on {correct} of {len(gold)} complaints.")

You matched the reference on 10 of 10 complaints.


### Demo — now let the computer follow a rule *we* wrote

Below is a **rule-based classifier**: a human (us) listed keywords that suggest danger. The computer just checks for them. Notice we had to think of every keyword ourselves.

<!-- cell-diagram:c14 -->
<p align="center"></p>

### Check your understanding (before running)
**Predict:** the keyword list below does *not* include the word "filthy" or "rude". What will the rule do with a complaint about a dirty or rude bus — call it Urgent or Routine? Is that the right call?

In [ ]:
# --- 1. DEFINE RULE CRITERIA ---
# This list contains specific words that a human expert identifies as signals for 'Urgent' safety risks.
# The system relies entirely on these predefined terms to make decisions.
URGENT_KEYWORDS = [
    "unsafe", "speeding", "dangerous", "brake",
    "accident", "lost", "fire", "injured", "bald tyres"
]

def rule_based(text):
    """
    Classifies a complaint based on the presence of safety keywords.

    Args:
        text (str): The raw text of the passenger complaint.

    Returns:
        str: 'Urgent' if any safety keyword is found (case-insensitive), otherwise 'Routine'.
    """
    t = text.lower()  # Normalize text to lowercase to ensure matching works regardless of capitalization
    return "Urgent" if any(k in t for k in URGENT_KEYWORDS) else "Routine"

# --- 2. APPLY THE RULE ---
# Run our rule-based function across the entire 'raw_text' column of our sample data.
sample["rule_label"] = sample["raw_text"].apply(rule_based)

# --- 3. EVALUATE PERFORMANCE ---
# Convert the boolean 'is_urgent' ground truth into matching strings ('Urgent'/'Routine') for comparison.
gold_series = sample["is_urgent"].map({True: "Urgent", False: "Routine"})

# Compare the rule's predictions against the reference (gold) labels and count the matches.
rule_correct = int((sample["rule_label"] == gold_series).sum())

# --- 4. DISPLAY RESULTS ---
print(f"The keyword rule matched {rule_correct} of {len(sample)} reference labels.")

# Show the text alongside the rule's decision and the actual reference for debugging and verification.
sample[["raw_text", "rule_label"]].assign(reference=gold_series)

The keyword rule matched 9 of 10 reference labels.


,raw_text,rule_label,reference
0,R7 never showed at the scheduled time again.,Routine,Routine
1,"Bus on R7 was 47 minutes late this morning, no...",Routine,Routine
2,Is R12 running during the public holiday?,Routine,Routine
3,The R7 vehicle had bald tyres and a broken door.,Urgent,Urgent
4,"Waited almost an hour for R9, completely unacc...",Routine,Routine
5,Driver on R9 was speeding and overtaking dange...,Urgent,Urgent
6,R14 never showed at the scheduled time again.,Routine,Routine
7,The R12 vehicle had bald tyres and a broken door.,Urgent,Urgent
8,Conductor on R3 was rude and shouting at an el...,Routine,Routine
9,Why did I pay 1865 Le when the board says the ...,Routine,Urgent


The rule is **brittle**: it only knows the words we thought of. A complaint that says *"the door flew open while moving"* is clearly urgent, but our rule misses it because "door" is not in the list. We could keep adding keywords forever — or we could let the computer **learn** which words matter from labelled examples. That is machine learning, and we start building it on Day 3.

### Exercise B — extend the rule (change one thing)

Add **one** keyword you think signals an urgent complaint, then re-run.

<!-- cell-diagram:c17 -->
<p align="center"></p>

### Check your understanding (before running)
You will add one keyword (e.g. `smoke`) to the urgent rule and re-run scoring.

**Predict:** Will one new keyword catch more R12 safety complaints, or add false alarms?

In [ ]:
# --- EXTENDING THE RULE-BASED SYSTEM ---

# 1. DEFINE IMPROVED CRITERIA
# To address integrity issues (like overcharging), we extend the original keyword list.
# Adding 'pay' allows us to catch financial or administrative urgency that 'safety' words missed.
URGENT_KEYWORDS_V2 = URGENT_KEYWORDS + ["pay"]

def rule_based_v2(text):
    """
    An updated classification rule that includes integrity/pricing keywords.

    Args:
        text (str): The passenger complaint text to evaluate.

    Returns:
        str: 'Urgent' if safety or integrity keywords are present, else 'Routine'.
    """
    t = text.lower()
    return "Urgent" if any(k in t for k in URGENT_KEYWORDS_V2) else "Routine"

# 2. GENERATE NEW PREDICTIONS
# Apply the updated logic to the same sample dataset to see the impact of our change.
new_label = sample["raw_text"].apply(rule_based_v2)

# 3. QUANTIFY THE CHANGE
# Identify how many classifications shifted from 'Routine' to 'Urgent' (or vice versa).
changed = int((new_label != sample["rule_label"]).sum())
print(f"Your new keyword changed the label on {changed} complaint(s).")

# 4. MEASURE ACCURACY GAIN
# Re-evaluate against the 'gold' reference labels to see if accuracy improved.
gold_series = sample["is_urgent"].map({True: "Urgent", False: "Routine"})
new_correct = int((new_label == gold_series).sum())

print(f"The updated rule now matches {new_correct} of {len(sample)} reference labels.")

Your new keyword changed the label on 1 complaint(s).
The updated rule now matches 10 of 10 reference labels.


## When should we **not** use AI here?

AI predicts patterns; it does not *know* facts. For a confirmed safety emergency (a crash, a fire), dispatch must act on the **report**, not on a model's guess. Keep this question alive all course: *what does it cost us if the model is wrong?*

## Check your understanding
1. In your own words, what is the difference between a rule-based system and a learning system?
2. Give one SLPTA task you would solve with a simple rule, and one you would solve by learning from examples. Why?
3. Name one dispatch decision where you would **not** rely on AI, and explain why.

4. **Predict:** We are going to add one keyword to the urgent list and re-run. Without running the code, will the number of correctly labelled complaints go up, down, or stay the same? Why might adding more keywords not always help?

### Your answers
*Double-click to edit this cell and type your answers here.*

1.
2.
3.

4.

## If you remember one thing today…

> **AI learns the rule from examples; traditional code follows the rule we wrote by hand.**

## Submission checklist
- [ ] Run every code cell successfully (top to bottom)
- [ ] Complete the exercise (fill every blank / make the requested change)
- [ ] Answer the **Check your understanding** questions in the markdown cell provided
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)